# 🎧 AI Support Envoy — GRPO Training

**OpenEnv Hackathon | Theme #3.1 World Modeling + Theme #1 Multi-Agent**

This notebook trains a compact LLM (`Qwen/Qwen2.5-0.5B-Instruct`) on AI Support Envoy using GRPO (Group Relative Policy Optimization).

The environment now supports both core tracks (`easy`, `medium`, `hard`) and an optional **frontier** track with governance-aware actions and tool evidence collection.

**Reward signal:** Dense shaped rewards based on:
- Correct categorization / prioritization
- Resolution quality (knowledge base alignment)
- SLA compliance
- Sentiment recovery (empathy detection)
- VIP customer awareness
- Governance-safe handoff and evidence-aware behavior (frontier)

> Runtime note: `frontier` fine-tuning/eval is optional in this notebook to keep T4 Colab runs manageable.

In [ ]:
import warnings
import logging
import os
import sys

# 🔧 FIX: Handle TRANSFORMERS_CACHE removal in transformers v4.48+
try:
    import transformers.utils.hub
    if not hasattr(transformers.utils.hub, 'TRANSFORMERS_CACHE'):
        from huggingface_hub.constants import HF_HUB_CACHE
        transformers.utils.hub.TRANSFORMERS_CACHE = HF_HUB_CACHE
except ImportError:
    pass

# Silence all warnings and library chatter
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('trl').setLevel(logging.ERROR)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('✅ Output Silenced: Only critical errors and progress bars will show.')

In [ ]:
# 1. Install dependencies
# Suppressing version conflicts that don't affect our project
!pip install -q -U pydantic click
!pip install -q openenv "trl>=0.12.0" "transformers<4.48.0" accelerate python-dotenv matplotlib mergekit llm-blender
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
print('\n✅ Dependencies installed! Please click "RESTART SESSION" if prompted above.')

In [ ]:
# Clone the environment repo
!git clone https://github.com/Puni2001/openenv-customer-support.git /content/openenv-customer-support
%cd /content/openenv-customer-support

In [ ]:
import sys
sys.path.insert(0, '/content/openenv-customer-support')

from src.customer_support_env import CustomerSupportEnv, Action, KNOWLEDGE_BASE
from tasks.grader import TaskGrader
print('Environment loaded successfully')

## 1. Baseline Evaluation (Before Training)

In [ ]:
import random

def random_agent_baseline(task_level: str, n_episodes: int = 20):
    """Random agent to establish baseline reward."""
    from src.customer_support_env import TicketCategory, Priority
    env = CustomerSupportEnv(task_level=task_level)
    episode_rewards = []

    for _ in range(n_episodes):
        obs = env.reset()
        total_reward = 0
        while not env.done:
            if task_level == 'easy':
                action = Action(action_type='categorize',
                                value=random.choice([c.value for c in TicketCategory]))
            elif task_level == 'medium':
                action = Action(action_type='prioritize',
                                value=random.choice([p.value for p in Priority]))
            else:
                action = Action(action_type='resolve', value='generic resolution attempt')
            obs, reward, done, _ = env.step(action)
            total_reward += reward
        episode_rewards.append(total_reward)

    return episode_rewards

print('Running random baseline...')
baseline_easy   = random_agent_baseline('easy',   20)
baseline_medium = random_agent_baseline('medium', 20)
baseline_hard   = random_agent_baseline('hard',   20)

print(f'Baseline easy:   avg={sum(baseline_easy)/len(baseline_easy):.4f}')
print(f'Baseline medium: avg={sum(baseline_medium)/len(baseline_medium):.4f}')
print(f'Baseline hard:   avg={sum(baseline_hard)/len(baseline_hard):.4f}')

## 2. Generate Training Dataset

In [ ]:
import json

SYSTEM_PROMPTS = {
    'easy':   'You are a customer support triage agent. Classify the ticket into exactly one category: technical, billing, account, feature_request, or complaint. Respond ONLY in JSON: {"action_type": "categorize", "value": "<category>", "reasoning": "<why>"}',
    'medium': 'You are a support ops agent. Set the correct priority: low, medium, high, or urgent. Respond ONLY in JSON: {"action_type": "prioritize", "value": "<priority>", "reasoning": "<why>"}',
    'hard':   'You are a support resolution specialist. Resolve the ticket using KB steps or escalate (if sentiment < -0.7 or category is complaint). Respond ONLY in JSON: {"action_type": "resolve|escalate", "value": "<resolution>", "reasoning": "<why>"}',
    'frontier': 'You are a governance-aware support agent. For high-risk cases collect evidence first via tool_call (policy_lookup, fraud_screen, kyc_verify, trust_safety_review), then choose resolve/escalate/human_review_required/legal_hold. Respond ONLY in JSON: {"action_type": "tool_call|resolve|escalate|human_review_required|legal_hold", "value": "<tool_or_decision>", "reasoning": "<why>"}',
}

def generate_dataset(task_level: str, n: int = 300):
    from src.customer_support_env import CustomerSupportEnv, KNOWLEDGE_BASE
    env = CustomerSupportEnv(task_level=task_level)
    samples = []
    system = SYSTEM_PROMPTS[task_level]
    for _ in range(n):
        obs = env.reset()
        t = obs.current_ticket
        if not t:
            continue

        # No label leakage in prompt.
        meta = [
            f'Ticket: {t.description}',
            f'Sentiment: {t.sentiment:.2f}',
            f'SLA: {obs.current_sla_status}',
            f'VIP: {t.is_vip}',
            f'Previous Contacts: {t.previous_contacts}',
        ]

        if task_level in ('hard', 'frontier'):
            kb = KNOWLEDGE_BASE.get(t.category.value, {})
            meta.append(f'KB Steps: {", ".join(kb.get("steps", []))}')

        if task_level == 'frontier':
            meta.append(f'High Risk Flags: {", ".join(t.high_risk_flags) if t.high_risk_flags else "none"}')
            meta.append(f'Governance Hint: {obs.governance_hint}')

        user = '\n'.join(meta)

        # Store GROUND TRUTH in payload for reward function.
        payload = {
            'category': t.category.value,
            'sentiment': t.sentiment,
            'priority': t.priority.value,
            'is_vip': t.is_vip,
            'previous_contacts': t.previous_contacts,
        }

        samples.append({
            'prompt': [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}],
            'ticket_payload': json.dumps(payload),
        })
    return samples

# Fast-track sample counts for T4 runtime.
train_easy = generate_dataset('easy', 100)
train_medium = generate_dataset('medium', 150)
train_hard = generate_dataset('hard', 200)
ENABLE_FRONTIER = False  # switch to True for optional frontier fine-tuning
train_frontier = generate_dataset('frontier', 120) if ENABLE_FRONTIER else []

print(
    f'✅ Datasets ready: easy={len(train_easy)}, medium={len(train_medium)}, '
    f'hard={len(train_hard)}, frontier={len(train_frontier)} (enabled={ENABLE_FRONTIER})'
)

## 3. Reward Functions

In [ ]:
import re
import json
from datetime import datetime, timedelta
from src.customer_support_env import CustomerSupportEnv, Action, Ticket, TicketCategory, Priority

def make_reward_fn(task_level):
    def reward_fn(completions, prompts=None, **kwargs):
        rewards = []
        ticket_payloads = kwargs.get('ticket_payload', [])
        
        for i, completion in enumerate(completions):
            try:
                # 1. Extract completion text
                if isinstance(completion, list) and len(completion) > 0:
                    text = completion[0].get('content', str(completion))
                else:
                    text = str(completion)
                
                # 2. Extract JSON action
                data = {}
                s, e = text.find('{'), text.rfind('}') + 1
                if s != -1:
                    data = json.loads(text[s:e])
                
                action = Action(
                    action_type=data.get('action_type', 'categorize'),
                    value=str(data.get('value', '')),
                    reasoning=str(data.get('reasoning', ''))
                )
                
                # 3. Get GROUND TRUTH from payload (REWARD-CONTEXT COUPLING FIX)
                payload = json.loads(ticket_payloads[i]) if i < len(ticket_payloads) else {}
                
                ticket = Ticket(
                    customer_id="RL_USER",
                    category=TicketCategory(payload.get('category', 'technical')),
                    description="Validation Ticket",
                    sentiment=float(payload.get('sentiment', 0.0)),
                    priority=Priority(payload.get('priority', 'medium')),
                    is_vip=bool(payload.get('is_vip', False)),
                    previous_contacts=int(payload.get('previous_contacts', 0)),
                    sla_deadline=datetime.now() + timedelta(hours=2)
                )

                # 4. Score using environment logic with TRUE ticket
                env = CustomerSupportEnv(task_level=task_level)
                reward_val, _ = env._calculate_reward(action, ticket)
                rewards.append(float(reward_val))
                
            except Exception as e:
                rewards.append(-0.5) # Penalty for failure
        return rewards
    return reward_fn


## 4. GRPO Training with Curriculum

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import GRPOTrainer, GRPOConfig

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
print(f'Loading {MODEL_ID} with Unsloth...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
    return_dict=True,
)

# SAFETY FIX: Move these outside the parenthesis
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    use_gradient_checkpointing='unsloth',
)

print('Model loaded safely')


In [ ]:
training_log = []  # for reward curves

curriculum = [('easy', train_easy), ('medium', train_medium), ('hard', train_hard)]
if ENABLE_FRONTIER:
    curriculum.append(('frontier', train_frontier))

for task_level, dataset in curriculum:
    print(f'\n=== Training: {task_level} ===')

    config = GRPOConfig(
        output_dir=f'./checkpoints/{task_level}',
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=5e-6,
        max_completion_length=200,
        num_generations=4,
        temperature=0.8,
        logging_steps=5,
        save_steps=50,
        report_to='none',
    )

    # 1. Define the callback
    from transformers import TrainerCallback
    class LiveMonitorCallback(TrainerCallback):
        def on_step_end(self, args, state, control, **kwargs):
            if state.global_step % 50 == 0 and state.global_step > 0:
                print(f"\n🚀 STEP {state.global_step}: THE AGENT IS THINKING...")
                model.eval()
                test_sample = dataset[0]['prompt']
                inputs = tokenizer.apply_chat_template(test_sample, return_tensors="pt", add_generation_prompt=True).to("cuda")
                with torch.no_grad():
                    outputs = model.generate(inputs, max_new_tokens=100)
                    print(tokenizer.decode(outputs[0], skip_special_tokens=True).split('assistant')[-1])
                model.train()

    # 2. Aggressive patch to fix the 'set' vs 'dict' error
    for m in [model, getattr(model, 'base_model', None), getattr(getattr(model, 'base_model', None), 'model', None)]:
        if m is not None: object.__setattr__(m, 'warnings_issued', {})

    # 3. Create the trainer with keyword arguments only
    trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=make_reward_fn(task_level),
        args=config,
        train_dataset=dataset,
        callbacks=[LiveMonitorCallback()]
    )

    result = trainer.train()
    training_log.append({'task': task_level, 'log_history': trainer.state.log_history})
    print(f'Done: {task_level}')


## 5. Reward Curves

If `ENABLE_FRONTIER=True`, an additional frontier curve will be appended automatically.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(15, 5))
fig.suptitle('AI Support Envoy — GRPO Training Reward Curves', fontsize=14, fontweight='bold')

colors = {'easy': '#22c55e', 'medium': '#f59e0b', 'hard': '#ef4444'}

for i, log in enumerate(training_log):
    ax = fig.add_subplot(1, 3, i+1)
    rewards = [e.get('reward', e.get('train_reward', 0)) for e in log['log_history'] if 'reward' in e or 'train_reward' in e]
    steps = list(range(len(rewards)))
    ax.plot(steps, rewards, color=colors[log['task']], linewidth=2)
    ax.fill_between(steps, rewards, alpha=0.15, color=colors[log['task']])
    ax.set_title(f"{log['task'].capitalize()} Task", fontweight='bold')
    ax.set_xlabel('Training Steps')
    ax.set_ylabel('Reward')
    ax.grid(True, alpha=0.3)
    ax.set_facecolor('#0a0f1e')
    fig.patch.set_facecolor('#111827')
    ax.tick_params(colors='white')
    ax.title.set_color('white')
    ax.xaxis.label.set_color('white')
    ax.yaxis.label.set_color('white')
    for spine in ax.spines.values(): spine.set_edgecolor('#334155')

plt.tight_layout()
plt.savefig('reward_curves.png', dpi=150, bbox_inches='tight', facecolor='#111827')
plt.show()
print('Saved: reward_curves.png')

## 6. Before vs After Evaluation

This section keeps the original core comparison (`easy`, `medium`, `hard`) for continuity with earlier artifacts. Use `evaluate_models.py --offline --tasks easy,medium,hard,frontier ...` for the current full scorecard pipeline.

In [ ]:
# Compare random baseline vs trained model with real measurements

def evaluate_trained_model(task_level: str, n_episodes: int = 20):
    env = CustomerSupportEnv(task_level=task_level)
    episode_rewards = []

    for _ in range(n_episodes):
        obs = env.reset()
        total_reward = 0.0
        max_steps = env.task_config['max_steps']
        step = 0

        while not env.done and step < max_steps:
            step += 1
            messages = [
                {'role': 'system', 'content': SYSTEM_PROMPTS[task_level]},
                {'role': 'user', 'content': (
                    f"Ticket: {obs.current_ticket.description}\n"
                    f"Sentiment: {obs.current_ticket.sentiment:.2f}\n"
                    f"SLA: {obs.current_sla_status}\n"
                    f"VIP: {obs.current_ticket.is_vip}\n"
                    f"Previous Contacts: {obs.current_ticket.previous_contacts}"
                )}
            ]

            inputs = tokenizer.apply_chat_template(
                messages,
                return_tensors='pt',
                add_generation_prompt=True
            ).to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    inputs,
                    max_new_tokens=120,
                    temperature=0.2,
                    do_sample=True,
                    pad_token_id=tokenizer.eos_token_id,
                )

            text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
            try:
                s, e = text.find('{'), text.rfind('}') + 1
                data = json.loads(text[s:e]) if s != -1 and e > s else {}
            except Exception:
                data = {}

            action = Action(
                action_type=str(data.get('action_type', 'request_info')),
                value=str(data.get('value', '')),
                reasoning=str(data.get('reasoning', '')),
            )
            obs, reward, _, _ = env.step(action)
            total_reward += reward

        episode_rewards.append(total_reward)

    return episode_rewards

print('Running trained-model evaluation...')
trained_easy   = evaluate_trained_model('easy', 20)
trained_medium = evaluate_trained_model('medium', 20)
trained_hard   = evaluate_trained_model('hard', 20)

results = {
    'before': {
        'easy':   sum(baseline_easy)/len(baseline_easy),
        'medium': sum(baseline_medium)/len(baseline_medium),
        'hard':   sum(baseline_hard)/len(baseline_hard),
    },
    'after': {
        'easy':   sum(trained_easy)/len(trained_easy),
        'medium': sum(trained_medium)/len(trained_medium),
        'hard':   sum(trained_hard)/len(trained_hard),
    }
}

print('=== Before vs After ===')
print(f'{"Task":<10} {"Before":>10} {"After":>10} {"Improvement":>12}')
print('-' * 45)
for task in ['easy', 'medium', 'hard']:
    before = results['before'][task]
    after  = results['after'][task]
    delta  = ((after - before) / abs(before)) * 100 if before != 0 else 0
    print(f'{task:<10} {before:>10.4f} {after:>10.4f} {delta:>+11.1f}%')

os.makedirs('results', exist_ok=True)
with open('results/baseline_vs_trained_colab.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved: results/baseline_vs_trained_colab.json')

## 7. Push to HuggingFace Hub

Upload the trained model and reward curves to HuggingFace.

### Optional: Generate latest offline benchmark pack from Colab runtime

```bash
python evaluate_models.py --offline --tasks easy,medium,hard,frontier --episodes 3 --seeds 41,42,43 --output results/final_baseline_vs_trained.md --trained-model offline_stub
python ablation_eval.py
```

This produces the same judge-facing artifacts used in the repository (`results/final_*`, `results/ablation_hack_penalty.json`).

In [ ]:
from huggingface_hub import login
login()  # enter your HF token

REPO_ID = 'punith2001/openenv-customer-support-model'
model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)
print(f'Model pushed to: https://huggingface.co/{REPO_ID}')